# Emotion-Aware Image Captioning with ViT-GPT2

## Features:
- Base caption generation using ViT-GPT2 (from BASE code)
- Emotion detection using DeepFace
- Grammatical emotion insertion using NLTK
- All original evaluation metrics preserved (BLEU, METEOR)
- Enhanced CSV outputs with emotion information

In [ ]:
import numpy as np 
import pandas as pd 
import os
from PIL import Image
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
import torchvision
from torchvision import transforms
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Tuple, Dict, Union, List
import transformers
import math
import random
import warnings
import timm
import time
import pickle

In [ ]:
# === ADDED: Install required libraries for BLEU, METEOR, and EMOTION DETECTION ===
!pip install nltk -q
!pip install deepface -q
!pip install tf-keras -q
!pip install opencv-python-headless -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from tqdm import tqdm

# Import DeepFace for emotion detection
from deepface import DeepFace

print("NLTK, evaluation libraries, and DeepFace loaded successfully!")

In [ ]:
image_folder = "/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images"
csv_file_path = "/kaggle/input/flickr-image-dataset/flickr30k_images/results.csv"

In [ ]:
os.listdir("/kaggle/input/flickr-image-dataset/flickr30k_images/")

## Utilities

In [ ]:
def clear_console():
    if os.name == 'nt':  
        os.system('cls')
    else:  
        os.system('clear')

## Hyperparams

In [ ]:
# === MODIFIED: Changed train_size from 0.9 to 0.8 (80% train, 20% test) ===
train_size = 0.8
bs = 16
img_size = 224

### CHANGE LIKE PAPER: context length changed from 256 to 50 (paper recommends 20-50)
ctx_length = 50

num_encoders_vit = 8
num_heads_vit = 4
ps = 16
c = 3
d_model_vit = ps**2*c
num_patches = (img_size*img_size)//(ps*ps)
d_model_gpt = 512
num_decoders_gpt = 8
num_heads_gpt = 8
softmax_denom_eps = 1e-9
device = "cuda" if torch.cuda.is_available() else "cpu"

### CHANGE LIKE PAPER: dropout changed from 0 to 0.5 (paper recommends 0.1-0.5)
attn_dropout = 0.5
mlp_dropout = 0.5
emb_dropout = 0.5

vit_kwargs = {
    "num_encoders" : num_encoders_vit,
    "num_heads": num_heads_vit,
    "num_patches": num_patches,
    "patch_size": ps,
    "channels": c,
    "d_model": d_model_vit,
    "pretrained_model_name": None,
    "device": device,
    "emb_dropout": emb_dropout,
    "mlp_dropout": mlp_dropout,
    "attn_dropout": attn_dropout
}

gpt_kwargs = {
    "d_model": d_model_gpt,
    "context_length": ctx_length,
    "num_decoders": num_decoders_gpt,
    "softmax_eps": softmax_denom_eps,
    "num_heads": num_heads_gpt,
    "device": device,
    "emb_dropout": emb_dropout,
    "mlp_dropout": mlp_dropout,
    "attn_dropout": attn_dropout
# should add ignore_index and vocab_size before sending to the model
}

config = {
    "vit_kwargs": vit_kwargs,
    "gpt_kwargs": gpt_kwargs,
    "device": device
}

## Tokenizer

In [ ]:
class Tokenizer:
    
    def __init__(self, texts: List[str], special_tokens_dict: Dict[str, str]) -> None:
        
        if special_tokens_dict is None:
            warnings.warn(f"'special_tokens_dict' has not been set, using default special_tokens_dict")
            self.special_tokens_dict = {
                "bos_token": "[BOS]",
                "eos_token": "[EOS]",
                "pad_token": "[PAD]"
            }
        else:
            assert 'pad_token' in special_tokens_dict, ValueError("'pad_token' key must be present in the 'special_tokens_dict' passed")
            self.special_tokens_dict = special_tokens_dict
        
        self.texts = texts
        self.pad_token = self.special_tokens_dict['pad_token']
        self.pad_token_id = None
        self._create_mapping()
    
    def _create_mapping(self) -> None:
        
        complete_text: str = sorted(list(set(". ".join(self.texts))))
        self.itos = {i:ch for i, ch in enumerate(complete_text)}
        self.stoi = {ch: i for i, ch in enumerate(complete_text)}

        for v in self.special_tokens_dict.values():
            self.itos[len(self.itos)] = v
            self.stoi[v] = len(self.stoi)

        if self.pad_token is not None:
            if self.stoi.get(self.pad_token, None):
                raise ValueError(f"{self.pad_token} is present in the vocabulary")
            self.pad_token_id = self.stoi.get(self.pad_token)
        
        self.vocab_size = len(self.itos)
    
    def encode(self, text: str, max_len: int, padding: bool=True) -> Dict[str, torch.Tensor]:
        
        words = text.split(' ')
        tokens = []
        for word in words:
            if word in self.special_tokens_dict.values():
                tokens.append(self.stoi[word])
            else:
                tokens.extend([self.stoi[ch] for ch in word])

        attn_mask = [1 for _ in range(len(tokens))]
        if padding:
            if self.pad_token is None:
                raise ValueError("padding cannot be done when 'pad_token' is not set")
            if len(tokens) < max_len: 
                pad_len = max_len - len(tokens)
                tokens += [self.pad_token_id]*pad_len
                attn_mask += [0]*pad_len
        
        return {
            "input_ids": torch.tensor(tokens[:max_len], requires_grad=False),
            "attention_mask": torch.tensor(tokens[:max_len], requires_grad=False)
        }
    
    def decode(self, token_ids: List[int]) -> str:
        
        return ''.join([self.itos[token] for token in token_ids])
    
    def get_vocab(self):
        return self.stoi

    def __call__(self, *args, **kwargs):
        return self.encode(*args, **kwargs)
    
    def save(self, file_path):
        with open(file_path, "wb") as f:
            pickle.dump(self, f)
    
    @staticmethod
    def load(file_path):
        with open(file_path, "rb") as f:
            return pickle.load(f)


class TokenizerHF:

    def __init__(self, tokenizer_name, special_tokens_dict=None) -> None:

        self.tokenizer = transformers.AutoTokenizer.from_pretrained(tokenizer_name)
        if special_tokens_dict is None:
           warnings.warn(f"'special_tokens_dict' has not been set, using default special_tokens_dict")
           self.tokenizer.add_special_tokens({
               "bos_token": "[BOS]",
               "eos_token": "[EOS]",
               "pad_token": "[PAD]"
           }) 
           self.vocab_size = self.tokenizer.vocab_size + 3
           self.pad_token = '[PAD]'
        else:
            assert 'pad_token' in special_tokens_dict, ValueError("'pad_token' key must be present in the 'special_tokens_dict' passed")                
            self.tokenizer.add_special_tokens(special_tokens_dict)
            self.vocab_size = self.tokenizer.vocab_size + len(special_tokens_dict)
            self.pad_token = special_tokens_dict['pad_token']

    def encode(self, text, max_len, padding=True) -> Dict[str, torch.Tensor]:
        return self.tokenizer(
            text,
            max_length=max_len,
            padding='max_length' if padding else True,
            truncation=True,
            return_tensors='pt'
        )

    def decode(self, token_ids) -> str:
        return self.tokenizer.decode(token_ids)
    
    def __call__(self, *args, **kwargs):
        return self.encode(*args, **kwargs)

    def get_vocab(self):
        return self.tokenizer.get_vocab()
    
    def save(self, file_path):
        with open(file_path, "wb") as f:
            pickle.dump(self, f)
    
    @staticmethod
    def load(file_path):
        with open(file_path, "rb") as f:
            return pickle.load(f)


## Data

In [ ]:
data = pd.read_csv(csv_file_path, delimiter="|")
data.drop_duplicates(subset=['image_name'], inplace=True)
data.drop(columns=' comment_number', axis=1, inplace=True)
data.reset_index(drop=True, inplace=True)
data.rename({" comment": "comment"}, axis=1, inplace=True)
data.iloc[:, 0] = image_folder + "/" + data.iloc[:, 0]
data['comment'] = '[BOS] ' + data['comment'] + ' [EOS]'
# tokenizer = Tokenizer(texts=data['comment'].tolist(), pad_token='/')
tokenizer = TokenizerHF(tokenizer_name="gpt2", special_tokens_dict={"bos_token": "[BOS]", "eos_token": "[EOS]", "pad_token": "[PAD]"})


class ImageCaptionDataset(Dataset):
    
    def __init__(self, dataframe: pd.DataFrame, image_size: int, context_length: int) -> None:
        
        assert dataframe.columns[0] == 'image_name', ValueError("The first column should be the path to the image")
        assert dataframe.columns[1] == "comment", ValueError("The second column should be named 'comment'")
        
        self.context_length = context_length
        self.df = dataframe
        self.transform = transforms.Compose([
            transforms.Resize(size=(image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
            
    def __len__(self) -> int:
        return self.df.shape[0]
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor]:
        
        image, text = Image.open(self.df.iloc[idx, 0]), self.df.iloc[idx, 1]
        image_tensor = self.transform(image)
        op = tokenizer(text, max_len=self.context_length+1)
        tokens, attention_mask = op['input_ids'].squeeze(), op['attention_mask'].squeeze()
        return image_tensor, tokens, attention_mask
    

def prepare_data(train_size: float, image_size: int, batch_size: int) -> Tuple[DataLoader]:
 
    idxs = set(range(data.shape[0]))
    train_idxs = random.sample(sorted(idxs), k=int(len(idxs)*train_size))
    test_idxs = list(idxs.difference(set(train_idxs)))

    train_data = data.copy(deep=True).iloc[train_idxs, :].reset_index(drop=True)
    test_data = data.copy(deep=True).iloc[test_idxs, :].reset_index(drop=True)   


    train_dataset = ImageCaptionDataset(
        dataframe = train_data,
        image_size = image_size,
        context_length = ctx_length
    )

    test_dataset = ImageCaptionDataset(
        dataframe = test_data,
        image_size = image_size,
        context_length = ctx_length
    )

    train_dl = DataLoader(
        dataset = train_dataset,
        batch_size = batch_size,
        shuffle = True,
        num_workers = 2
    )

    test_dl = DataLoader(
        dataset = test_dataset,
        batch_size = batch_size,
        shuffle = False
    )
    
    # === ADDED: Store test_data globally for evaluation later ===
    global test_data_df
    test_data_df = test_data

    return train_dl, test_dl

## ViT

In [ ]:
class PatchEmbeddings(nn.Module):
    
    """Class for creating the patches of an image using a convolutional layer"""

    def __init__(self, config) -> None:
        
        super().__init__()
        self.conv_patch_layer = nn.Conv2d(in_channels = config['channels'],
                                         out_channels = config['d_model'],
                                         kernel_size = config['patch_size'],
                                         stride = config['patch_size'])
        self.flatten = nn.Flatten(start_dim=2, end_dim=3)
        
    
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        
        # images -> B, C, H, W
        patched_images = self.conv_patch_layer(images) # B, D_MODEL, IMG_SIZE/PS, IMG_SIZE/PS
        flattened_patches = self.flatten(patched_images) # B, D_MODEL, N
        permuted_patches = flattened_patches.permute(0, 2, 1) # B, N, D_MODEL
        return permuted_patches


class ViTEmbedding(nn.Module):
    
    """Creates the input embeddings for the ViT class by combining both patch and positional embeddings"""

    def __init__(self, config) -> None:
        
        super().__init__()
        self.class_token_embedding = nn.Parameter(
            data = torch.randn(size=(1, 1, config['d_model'])),
            requires_grad = True
        )
        self.positional_embedding = nn.Parameter(
            data = torch.randn(size=(1, config['num_patches']+1, config["d_model"])),
            requires_grad = True
        )
        self.patch_embeddings_layer = PatchEmbeddings(config)
        self.dropout = nn.Dropout(p=config['emb_dropout'])
        
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # images -> B, C, H, W
        patch_embeddings = self.patch_embeddings_layer(images) # B, NUM_PATCHES, D_MODEL
        patch_embeddings_with_class_token = torch.cat(
            tensors = (self.class_token_embedding.repeat(patch_embeddings.shape[0], 1, 1), patch_embeddings), 
            dim = 1
        ) # B, NUM_PATCHES+1, D_MODEL
        return self.dropout(patch_embeddings_with_class_token + self.positional_embedding)
    

class MSABlock(nn.Module):
    
    """Multihead Self attention block of the decoder. Uses PyTorch's inbuild MHA layer for calculating the attention sub-blocks output"""

    def __init__(self, config) -> None:
        
        super().__init__()
        
        self.attn_block = nn.MultiheadAttention(
            embed_dim = config["d_model"],
            num_heads = config["num_heads"],
            batch_first = True,
            dropout=config['attn_dropout']
        )
        self.layer_norm = nn.LayerNorm(normalized_shape=config["d_model"])
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        # x -> B, NUM_PATCHES+1, D_MODEL
        attn_output, _ = self.attn_block(x, x, x)
        return self.layer_norm(x + attn_output) 
    
class MLPBlock(nn.Module):
    
    """FFN block of the transformer architecture"""

    def __init__(self, config) -> None:
        
        super().__init__()
        d_model = config["d_model"]
        self.dense_net = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.GELU(),
            nn.Dropout(p=config['mlp_dropout']),
            nn.Linear(d_model*4, d_model)
        )
        
        self.layer_norm = nn.LayerNorm(normalized_shape=d_model)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.layer_norm(x + self.dense_net(x))

    
class EncoderBlock(nn.Module):
    
    """Encoder block which combines both the MSA and MLP blocks"""

    def __init__(self, config) -> None:
        
        super().__init__()
        self.msa_block = MSABlock(config)
        self.mlp_block = MLPBlock(config)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.mlp_block(self.msa_block(x))
    

class Encoder(nn.Module):
    
    """The encoder backbone of the ViT architecture, made up of 'n' encoder blocks"""

    def __init__(self, config) -> None: 
        
        super().__init__()
        self.blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config["num_encoders"])])
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        for block in self.blocks:
            x = block(x)
        
        return x
    
class ViT(nn.Module):
    
    """The ViT class which puts together the embeddings and the encoder. Returns the representation of the image usign the [CLS] token"""

    def __init__(self, config) -> None:
        
        super().__init__()
        
        self.embedding_layer = ViTEmbedding(config)
        self.encoder = Encoder(config)
    
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # B, C, H, W
        embeddings = self.embedding_layer(images) # B, NUM_PATCHES+1, D_MODEL
        encoded_vectors = self.encoder(embeddings) # B, NUM_PATCHES+1, D_MODEL
        return encoded_vectors[:, 0, :]

## GPT

In [ ]:
class GPTEmbedding(nn.Module):
    
    """Embedding class for the GPT decoder"""

    def __init__(self,config) -> None:
        
        super().__init__()
        self.token_embedding = nn.Embedding(
            num_embeddings = config["vocab_size"],
            embedding_dim = config["d_model"]
        )
        
        self.positional_encoding = nn.Parameter(
            data = torch.randn(size=(1, config["context_length"], config["d_model"])),
            requires_grad = True
        )
        self.dropout = nn.Dropout(p=config['emb_dropout'])
    
    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        
        # tokens -> B, CTX_LENGTH
        token_embeddings = self.token_embedding(tokens)
        return self.dropout(self.positional_encoding[:, :tokens.shape[1], :] + token_embeddings)


class CausalSelfAttnBlock(nn.Module):
    
    """Causal self attention class - does not use PyTorch's inbuild masked MHA, since it does not accomodate both causal mask and padding mask. 
       Safe softmax has been used instead of PyTorch's softmax, to bring numerical stability in the calculations.
    """

    def __init__(self, config) -> None:
        
        super().__init__()
        assert config["d_model"] % config["num_heads"] == 0, ValueError(f"{config['d_model']} d_model should be exactly divisible by {config['num_heads']} num_heads")
        
        self.d_model = config["d_model"]
        self.head_dim = config["d_model"] // config["num_heads"]
        self.num_heads = config["num_heads"]
        self.softmax_eps = config["softmax_eps"]
        
        self.projection_layer = nn.Linear(self.d_model, self.d_model*3)
        self.out_layer = nn.Linear(self.d_model, self.d_model)
        self.layer_norm = nn.LayerNorm(normalized_shape=self.d_model)
        self.attn_dropout = nn.Dropout(p=config['attn_dropout'])

    def _safe_softmax(self, x: torch.Tensor) -> torch.Tensor:
        
        num = torch.exp(x)
        denom = torch.exp(x).sum(dim=-1, keepdims=True) + self.softmax_eps
        return num/denom
    
    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        
        # x -> B, CTX_LENGTH, D_MODEL
        # attn_mask -> B, CTX_LENGTH, CTX_LENGTH
        B, CTX_LENGTH = x.shape[0], x.shape[1]
        q, k, v = self.projection_layer(x).split(self.d_model, dim=2) # B, CTX_LENGTH, D_MODEL
        q = q.view(B, CTX_LENGTH, self.num_heads, self.head_dim).transpose(1, 2) # B, NUM_HEADS, CTX_LENGTH, HEAD_DIM
        k = k.view(B, CTX_LENGTH, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, CTX_LENGTH, self.num_heads, self.head_dim).transpose(1, 2)
        
        q_k_prod = (q @ k.transpose(2, 3)) + attn_mask.unsqueeze(1) # B, NUM_HEADS, CTX_LENGTH, CTX_LENGTH
        wts = self._safe_softmax(q_k_prod / math.sqrt(self.head_dim)) # B, NUM_HEADS, CTX_LENGTH, CTX_LENGTH
        wts = self.attn_dropout(wts)
        attn_outputs = wts @ v # B, NUM_HEADS, CTX_LENGTH, HEAD_DIM
        y = attn_outputs.transpose(1, 2).contiguous().view(B, CTX_LENGTH, -1)
        return self.layer_norm(x + self.out_layer(y))
        
        

class CrossAttnBlock(nn.Module):
    
    """Cross attention block - similar version of Causal self attention block, but with the key and value tensors being the encoding of the image from ViT"""
    
    def __init__(self, config) -> None:
        
        super().__init__()

        assert config["d_model"]%config["num_heads"]==0, ValueError(f"{config['d_model']} d_model must be divisible by {config['num_heads']} num_heads")

        self.d_model = config['d_model']
        self.num_heads = config['num_heads']
        self.head_dim = config['d_model'] // config['num_heads']
        self.q_proj = nn.Linear(self.d_model, self.d_model)
        self.k_proj = nn.Linear(self.d_model, self.d_model)
        self.v_proj = nn.Linear(self.d_model, self.d_model)
        self.projection_layer = nn.Linear(self.d_model, self.d_model)
        self.layer_norm = nn.LayerNorm(normalized_shape=self.d_model)
        self.attn_dropout = nn.Dropout(p=config['attn_dropout'])
    
    def forward(self, x: torch.Tensor, image_encoding: torch.Tensor) -> torch.Tensor:
        
        # image_encoding (key, value)-> B, 1, D_MODEL
        # x (query)-> B, CTX_LENGTH, D_MODEL 
        
        # q_k_prod -> query @ key.transpose(1, 2) -> B, CTX_LENGTH, 1
        # q_k_prod @ value -> B, CTX_LENGTH, D_MODEL

        B, CTX_LENGTH, _ = x.shape        

        q = self.q_proj(x).view(B, CTX_LENGTH, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # B, NUM_HEADS, CTX_LENGTH, HEAD_DIM
        k = self.k_proj(image_encoding).view(B, 1, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # B, NUM_HEADS, 1, HEAD_DIM
        v = self.v_proj(image_encoding).view(B, 1, self.num_heads, self.head_dim).permute(0, 2, 1, 3) # B, NUM_HEADS, 1, HEAD_DIM

        wts = F.softmax((q @ k.transpose(2, 3)) / math.sqrt(self.head_dim), dim=-1) # B, NUM_HEADS, CTX_LENGTH, 1
        wts = self.attn_dropout(wts)
        y = wts @ v # B, NUM_HEADS, CTX_LENGTH, HEAD_DIM
        y = y.transpose(1, 2).contiguous().view(B, CTX_LENGTH, -1) # B, CTX_LENGTH, D_MODEL
        return self.layer_norm(x + self.projection_layer(y))
    
class GPTDecoderBlock(nn.Module):

    """The class which puts together Causal, Cross and FFN blocks together"""

    def __init__(self, config) -> None:
        
        super().__init__()
        self.csa_block = CausalSelfAttnBlock(config)
        self.cross_attn_block = CrossAttnBlock(config)
        self.mlp_block = MLPBlock(config)
    
    def forward(self, x: torch.Tensor, image_encoding: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        
        csa_out = self.csa_block(x, attn_mask)
        cross_out = self.cross_attn_block(csa_out, image_encoding)
        mlp_out = self.mlp_block(cross_out)
        return mlp_out

class GPTDecoder(nn.Module):
    
    """The backbone of the caption generator, made up of 'n' decoder blocks"""

    def __init__(self, config) -> None:
        
        super().__init__()
        self.decoder_blocks = nn.ModuleList([GPTDecoderBlock(config) for _ in range(config["num_decoders"])])
    
    def forward(self, x: torch.Tensor, image_encoding: torch.Tensor, attn_mask: torch.Tensor) -> torch.Tensor:
        
        for block in self.decoder_blocks:
            x = block(x, image_encoding, attn_mask)
        
        return x
    
class GPT(nn.Module):
    
    """The caption generator part of the system. Puts together everything"""

    def __init__(self, config) -> None:
        
        super().__init__()
        self.device = config["device"]
        self.context_length = config["context_length"]
        self.softmax_eps = config["softmax_eps"]
        self.embedding = GPTEmbedding(config)
        self.decoder = GPTDecoder(config)
        self.cls_head = nn.Linear(config["d_model"], config["vocab_size"])
        # self.cls_head.weight = self.embedding.token_embedding.weight
        # removed weight tying as it lead to slower convergence
        self.ignore_index = config["ignore_index"]
    
    def _create_mask(self, context_length: int, attn_mask: torch.Tensor) -> torch.Tensor:
        
        mask = torch.triu(
            input = torch.ones(size=(context_length, context_length), requires_grad = False)*float("-inf"),
            diagonal = 1
        ).unsqueeze(0).repeat(attn_mask.shape[0], 1, 1)
        mask = mask.to(self.device)
        for i in range(mask.shape[0]):
            mask[i, attn_mask[i].logical_not(), :] = float("-inf")
        return mask # B, CTX_LENGTH, CTX_LENGTH
        
    
    def forward(self, tokens: torch.Tensor, image_encoding: torch.Tensor, attn_mask: torch.Tensor, targets: torch.Tensor=None) -> Tuple[torch.Tensor]:
        
        # tokens -> B, CTX_LENGTH
        # image_encoding -> B, 1, D_MODEL
        # attn_mask -> B, CTX_LENGTH
        
        embeddings = self.embedding(tokens) # B, CTX_LENGTH, D_MODEL
        mask = self._create_mask(tokens.shape[1], attn_mask)
        decoder_out = self.decoder(embeddings, image_encoding, mask) # B, CTX_LENGTH, D_MODEL
        logits = self.cls_head(decoder_out) # B, CTX_LENGTH, VOCAB_SIZE
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.reshape(-1), ignore_index=self.ignore_index)
        
        return logits, loss

## Image Caption Model

In [ ]:
class ImageCaptionModel(nn.Module):
    
    """This class is the main class. Puts together both ViT and GPT. Lot more useful functions need to be added if someone uses them"""
    
    def __init__(self, config) -> None:
        
        super().__init__()
        
        self.device = config['device']
        self.is_vit_pretrained = False
        if config['vit_kwargs']["pretrained_model_name"] is not None:
            self.is_vit_pretrained = True
            self.vit = timm.create_model(
                model_name = config['vit_kwargs']["pretrained_model_name"],
                pretrained=True,
                num_classes = 0,
                global_pool = 'avg'
            )
            config["vit_kwargs"]["d_model"] = self.vit.embed_dim
        else:   
            self.vit = ViT(config['vit_kwargs'])
        self.gpt = GPT(config['gpt_kwargs'])
        self.dimension_mapping_layer = nn.Linear(config["vit_kwargs"]['d_model'], config["gpt_kwargs"]['d_model'])
        
    def forward(self, image: torch.Tensor, tokens: torch.Tensor, attn_mask: torch.Tensor, targets: torch.Tensor=None) -> Tuple[torch.Tensor]:
        
        image_encoding = self.vit(image)
        dimension_mapped_image_encoding = self.dimension_mapping_layer(image_encoding[:, None, :])
        return self.gpt(tokens, dimension_mapped_image_encoding, attn_mask, targets)

    
    @torch.inference_mode()
    def generate(self, 
                 image: torch.Tensor, 
                 sos_token: int,
                 eos_token: int,
                 max_len: int=40) -> List[int]:

        image_encoding: torch.Tensor = self.vit(image)
        dimension_mapped_image_encoding = self.dimension_mapping_layer(image_encoding[:, None, :])

        tokens = torch.tensor(data=[[sos_token]], requires_grad=False).to(self.device)
        attn_mask = torch.tensor(data=[[1]], requires_grad=False).to(self.device)

        while tokens.shape[1]<max_len and tokens[0, -1]!=eos_token:
            logits, _ = self.gpt(tokens, dimension_mapped_image_encoding, attn_mask, None) # 1, N+1, D_MODEL
            next_token = torch.argmax(logits[0, -1, :], dim=0).item()

            next_token_tensor = torch.tensor([[next_token]], requires_grad=False, device=self.device)
            tokens = torch.cat(
                (tokens, next_token_tensor),
                dim = -1
            )

            next_mask_tensor = torch.tensor([[1]], requires_grad=False, device=self.device)
            attn_mask = torch.cat(
                (attn_mask, next_mask_tensor),
                dim = -1
            )
        
        return list(tokens[0])
    
    @classmethod
    def from_pretrained(cls, checkpoint, device):

        if not os.path.exists(checkpoint):
            raise FileNotFoundError(f"{checkpoint} does not exist")

        cp = torch.load(checkpoint, map_location=device)
        cp['model_config']['device'] = device
        cp['model_config']['vit_kwargs']['device'] = device
        cp['model_config']['gpt_kwargs']['device'] = device

        model = cls(cp['model_config'])
        model.load_state_dict(cp['model_state_dict'])
        model = model.to(device)
        return model

## === ADDED: Emotion Detection and NLTK Insertion Functions ===

In [ ]:
# === EMOTION DETECTION AND GRAMMATICAL INSERTION ===

def detect_emotion_deepface(image_path):
    """
    Detect emotion using DeepFace from the image.
    Returns: emotion string (e.g., 'happy', 'sad', 'neutral')
    """
    try:
        # Analyze emotion
        result = DeepFace.analyze(img_path=image_path, actions=['emotion'], enforce_detection=False)
        
        # Extract dominant emotion
        if isinstance(result, list):
            emotion = result[0]['dominant_emotion']
        else:
            emotion = result['dominant_emotion']
        
        return emotion.lower()
    except Exception as e:
        print(f"[Warning] Emotion detection failed: {e}")
        return 'neutral'  # Default to neutral if detection fails


def insert_emotion_in_caption(caption, emotion):
    """
    Insert emotion adjective in grammatically correct position using NLTK.
    Logic: Insert emotion adjective before the first noun.
    """
    # Map emotions to descriptive adjectives
    emotion_adjectives = {
        'happy': 'joyful',
        'sad': 'melancholic',
        'angry': 'angry',
        'surprise': 'surprised',
        'fear': 'fearful',
        'disgust': 'disgusted',
        'neutral': 'calm'
    }
    
    emotion_adj = emotion_adjectives.get(emotion, emotion)
    
    # Tokenize and POS tag
    try:
        tokens = word_tokenize(caption)
        tagged = pos_tag(tokens)
        
        # Find first noun and insert emotion before it
        inserted = False
        new_tokens = []
        
        for i, (word, tag) in enumerate(tagged):
            if not inserted and tag.startswith('NN'):  # NN, NNS, NNP, NNPS
                new_tokens.append(emotion_adj)
                inserted = True
            new_tokens.append(word)
        
        # If no noun found, append emotion at the end
        if not inserted:
            new_tokens.insert(1, emotion_adj)  # Insert after first word
        
        return ' '.join(new_tokens)
    
    except Exception as e:
        print(f"[Warning] NLTK insertion failed: {e}")
        return f"{emotion_adj} {caption}"  # Fallback: prepend emotion


def generate_emotion_aware_caption(image_path, base_caption):
    """
    Generate emotion-aware caption by:
    1. Detecting emotion from image
    2. Inserting emotion in base caption using NLTK
    
    Returns: (emotion, emotion_aware_caption)
    """
    # Detect emotion
    emotion = detect_emotion_deepface(image_path)
    
    # Insert emotion in caption
    emotion_aware_caption = insert_emotion_in_caption(base_caption, emotion)
    
    return emotion, emotion_aware_caption

## Trainer

In [ ]:
train_dl, test_dl = prepare_data(train_size, img_size, bs)
config['gpt_kwargs']['vocab_size'] = tokenizer.vocab_size
config['gpt_kwargs']['ignore_index'] = tokenizer.get_vocab()[tokenizer.pad_token]

In [ ]:
class Trainer:

    def __init__(self, model_config, train_config, dls, tokenizer) -> None:
        
        self.device = train_config['device']
        self.model = ImageCaptionModel(model_config).to(self.device)
        self.train_config = train_config
        self.model_config = model_config
        self.train_dl, self.test_dl = dls
        self.metrics = pd.DataFrame(columns=["epoch", "train_loss", "test_loss", "train_perplexity", "test_perplexity", "elapsed_time"])
        self.tokenizer = tokenizer
        self.transform = transforms.Compose([
            transforms.Resize(size=(img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def fit(self):
        start_time = time.time()
        # train_config has to atleast consist of epochs and freeze_epochs
        if self.model.is_vit_pretrained and self.train_config["freeze_epochs"]>0:
            for p in self.model.vit.parameters():
                p.requires_grad = False
        
        self.optimizer = torch.optim.Adam([
                {"params": self.model.vit.parameters(), "lr": 0}, # setting to 0 will not update the frozen params
                {"params": self.model.dimension_mapping_layer.parameters(), "lr": self.train_config['lr']},
                {"params": self.model.gpt.parameters(), "lr": self.train_config['lr']}
            ],
            weight_decay=self.train_config['weight_decay']
        )

        for epoch in range(self.train_config["freeze_epochs"]):
            
            train_loss, train_perplexity = self._train()
            test_loss, test_perplexity = self._eval()
            elapsed_time = time.time() - start_time
            new_row = pd.DataFrame(data={
                "epoch": [epoch+1],
                "train_loss": [train_loss],
                "test_loss": [test_loss],
                "elapsed_time": [elapsed_time],
                "train_perplexity": [train_perplexity],
                "test_perplexity": [test_perplexity]
            })
            self.metrics = pd.concat([self.metrics, new_row], axis=0, ignore_index=True)

            clear_console()
            print(self.metrics.to_string(index=False))
            
        
        if self.model.is_vit_pretrained and self.train_config["freeze_epochs"]>0:
            for p in self.model.vit.parameters():
                p.requires_grad = True
        
        self.optimizer.param_groups[0]['lr'] = self.train_config['lr']  # unfreeze vit params

        for epoch in range(self.train_config["freeze_epochs"], self.train_config["epochs"]):
            train_loss, train_perplexity = self._train()
            test_loss, test_perplexity = self._eval()
            elapsed_time = time.time() - start_time
            new_row = pd.DataFrame(data={
                "epoch": [epoch+1],
                "train_loss": [train_loss],
                "test_loss": [test_loss],
                "elapsed_time": [elapsed_time],
                "train_perplexity": [train_perplexity],
                "test_perplexity": [test_perplexity]
            })
            self.metrics = pd.concat([self.metrics, new_row], axis=0, ignore_index=True)

            clear_console()
            print(self.metrics.to_string(index=False))
            
        return self.metrics

    def _train(self):

        self.model.train()
        total_loss = 0
        for image, tokens, attn_mask in self.train_dl:
        
            input_tokens, target_tokens = tokens[:, :-1], tokens[:, 1:]
            attn_mask = attn_mask[:, :-1]
            image, input_tokens, target_tokens, attn_mask = image.to(self.device), input_tokens.to(self.device), \
                                                            target_tokens.to(self.device), attn_mask.to(self.device)
            _, loss = self.model(image, input_tokens, attn_mask, target_tokens)
            total_loss += loss.item()
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
        
        avg_loss = total_loss / len(self.train_dl)
        train_perplexity = torch.exp(torch.tensor(avg_loss))
        return avg_loss, train_perplexity.item()

    
    def _eval(self):

        self.model.eval()
        total_loss = 0
        for image, tokens, attn_mask in self.test_dl:
        
            input_tokens, target_tokens = tokens[:, :-1], tokens[:, 1:]
            attn_mask = attn_mask[:, :-1]
            image, input_tokens, target_tokens, attn_mask = image.to(self.device), input_tokens.to(self.device), \
                                                            target_tokens.to(self.device), attn_mask.to(self.device)
            
            _, loss = self.model(image, input_tokens, attn_mask, target_tokens)
            total_loss += loss.item()
        
        avg_loss = total_loss / len(self.test_dl)
        test_perplexity = torch.exp(torch.tensor(avg_loss))
        return avg_loss, test_perplexity.item()
    

    def inference(self, image_path, max_len) -> str:
        image_tensor = self.transform(Image.open(image_path)).unsqueeze(0)
        tokens = self.model.generate(image_tensor, 
                                    sos_token=self.tokenizer.get_vocab()['[BOS]'],
                                    eos_token=self.tokenizer.get_vocab()['[EOS]'],
                                    max_len=max_len)
        return self.tokenizer.decode(token_ids=[token.item() for token in tokens])


    def save(self, file_path):
        
        checkpoint = {
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "train_config": self.train_config,
            "model_config": self.model_config
        }

        torch.save(checkpoint, file_path)

    def plot_metrics(self):
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.plot(self.metrics['epoch'], self.metrics['train_loss'], label='Train Loss')
        plt.plot(self.metrics['epoch'], self.metrics['test_loss'], label='Test Loss')
        plt.xlabel('epoch')
        plt.ylabel('loss')
        plt.title('Training and Test Loss Over Epochs')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(self.metrics['epoch'], self.metrics['train_perplexity'], label='Train Perplexity')
        plt.plot(self.metrics['epoch'], self.metrics['test_perplexity'], label='Test Perplexity')
        plt.xlabel('epoch')
        plt.ylabel('perplexity')
        plt.title('Training and Test Perplexity Over Epochs')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()

In [ ]:
train_config = {
        "epochs": 20,
        "freeze_epochs": 2,
        "lr": 2e-4,
        "device": device,
        "weight_decay": 1e-6
    }

config['device'] = device
config['gpt_kwargs']['device'] = device
config['vit_kwargs']['device'] = device

# === MODIFIED: Changed from "vit_tiny_patch16_224" to "vit_base_patch16_224" ===
config["vit_kwargs"]["pretrained_model_name"] = "vit_base_patch16_224"

trainer = Trainer(model_config=config,
                  train_config=train_config,
                  dls=(train_dl, test_dl),
                  tokenizer=tokenizer)

metrics = trainer.fit()

In [ ]:
trainer.plot_metrics()

In [ ]:
trainer.save("/kaggle/working/image_caption_model.pt")
print("Model saved!")

## Test on 5 Random Sample Images (Original Code - Kept As Is)

In [ ]:
# Test on 5 images (same dataset) - ORIGINAL CODE KEPT AS IS

# If you're running in a fresh Kaggle session, set these and the checkpoint path.
image_folder = "/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images"
csv_file_path = "/kaggle/input/flickr-image-dataset/flickr30k_images/results.csv"
checkpoint_path = "/kaggle/working/image_caption_model.pt"  # change if needed

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Ensure we have a trained/loaded model + tokenizer
if "trainer" not in globals():
    print("'trainer' not found. Loading checkpoint...")

    # Recreate tokenizer exactly like training
    tokenizer = TokenizerHF(
        tokenizer_name="gpt2",
        special_tokens_dict={"bos_token": "[BOS]", "eos_token": "[EOS]", "pad_token": "[PAD]"}
    )

    cp = torch.load(checkpoint_path, map_location=device)
    model = ImageCaptionModel(cp["model_config"]).to(device)
    model.load_state_dict(cp["model_state_dict"])
    model.eval()

    # Same transform as training
    img_size = 224
    transform = transforms.Compose([
        transforms.Resize(size=(img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
else:
    model = trainer.model
    tokenizer = trainer.tokenizer
    transform = trainer.transform
    model.eval()

# Load CSV and make full image paths exactly like training
if "data" not in globals():
    data = pd.read_csv(csv_file_path, delimiter="|")
    data.drop_duplicates(subset=["image_name"], inplace=True)
    data.drop(columns=" comment_number", axis=1, inplace=True)
    data.reset_index(drop=True, inplace=True)
    data.rename({" comment": "comment"}, axis=1, inplace=True)
    data.iloc[:, 0] = image_folder + "/" + data.iloc[:, 0]
    data["comment"] = "[BOS] " + data["comment"] + " [EOS]"

@torch.inference_mode()
def caption_one(image_path, max_len=40):
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)

    token_ids = model.generate(
        x,
        sos_token=tokenizer.get_vocab()["[BOS]"],
        eos_token=tokenizer.get_vocab()["[EOS]"],
        max_len=max_len,
    )

    text = tokenizer.decode([t.item() for t in token_ids])
    return text.replace("[BOS]", "").replace("[EOS]", "").replace("[PAD]", "").strip()

# Pick and caption 5 images
samples = data.sample(5, random_state=42)

for i, row in enumerate(samples.itertuples(index=False), start=1):
    img_path = row.image_name
    gt = row.comment.replace("[BOS]", "").replace("[EOS]", "").strip()

    pred = caption_one(img_path, max_len=40)

    print(f"\n=== Sample {i} ===")
    print("Image:", img_path)
    print("GT  :", gt)
    print("Pred:", pred)

    img = Image.open(img_path).convert("RGB")
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(pred)
    plt.show()

## === EVALUATION WITH EMOTION AWARENESS ===

### Changes:
1. `evaluation_results.csv` - Added "detected_emotion" and "emotion_aware_caption" columns
2. `evaluation_summary.csv` - UNCHANGED (only BASE caption metrics)
3. `results2.csv` - NEW file for custom images with emotion (no scoring)

In [ ]:
# === MODIFIED: Evaluate BLEU-1,2,3,4 and METEOR on the 20% test dataset WITH EMOTION ===

print("="*60)
print("EVALUATION ON 20% TEST DATASET (WITH EMOTION DETECTION)")
print("="*60)

# Use the test_data_df that was stored during data preparation
print(f"\nTest set size: {len(test_data_df)} images")

# Prepare lists for evaluation
all_references = []  # List of reference captions (ground truth)
all_hypotheses = []  # List of predicted BASE captions (no emotion)
evaluation_results = []  # Store individual results with emotion

# Smoothing function for BLEU (handles short sentences)
smoothing = SmoothingFunction().method1

print("\nGenerating captions for test set... (this may take a while)")

model.eval()

for idx in tqdm(range(len(test_data_df)), desc="Evaluating"):
    row = test_data_df.iloc[idx]
    img_path = row['image_name']
    
    # Ground truth caption (remove special tokens)
    gt_caption = row['comment'].replace("[BOS]", "").replace("[EOS]", "").strip()
    
    # Generate BASE prediction (for scoring)
    try:
        base_caption = caption_one(img_path, max_len=40)
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        base_caption = ""
    
    # === ADDED: Detect emotion and create emotion-aware caption ===
    try:
        emotion, emotion_aware_caption = generate_emotion_aware_caption(img_path, base_caption)
    except Exception as e:
        print(f"Error detecting emotion for {img_path}: {e}")
        emotion = "neutral"
        emotion_aware_caption = base_caption
    
    # Tokenize BASE caption for BLEU/METEOR calculation (emotion not included)
    gt_tokens = word_tokenize(gt_caption.lower())
    pred_tokens = word_tokenize(base_caption.lower())
    
    # Store for corpus BLEU (BASE captions only)
    all_references.append([gt_tokens])
    all_hypotheses.append(pred_tokens)
    
    # Calculate individual METEOR score (BASE caption only)
    try:
        individual_meteor = meteor_score([gt_tokens], pred_tokens)
    except:
        individual_meteor = 0.0
    
    # === MODIFIED: Store results with emotion columns ===
    evaluation_results.append({
        'image_path': img_path,
        'ground_truth': gt_caption,
        'prediction': base_caption,
        'meteor_score': individual_meteor,
        'detected_emotion': emotion,  # NEW COLUMN
        'emotion_aware_caption': emotion_aware_caption  # NEW COLUMN
    })

print("\nCaption generation complete!")

In [ ]:
# === UNCHANGED: Calculate and display average scores (BASE captions only) ===

print("\n" + "="*60)
print("CALCULATING EVALUATION METRICS (BASE CAPTIONS ONLY)")
print("="*60)

# Calculate corpus BLEU scores (BASE captions)
bleu1 = corpus_bleu(all_references, all_hypotheses, weights=(1, 0, 0, 0), smoothing_function=smoothing)
bleu2 = corpus_bleu(all_references, all_hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
bleu3 = corpus_bleu(all_references, all_hypotheses, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothing)
bleu4 = corpus_bleu(all_references, all_hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)

# Calculate average METEOR score (BASE captions)
avg_meteor = np.mean([r['meteor_score'] for r in evaluation_results])

print("\n" + "-"*40)
print("AVERAGE SCORES ON TEST SET (BASE CAPTIONS)")
print("-"*40)
print(f"BLEU-1: {bleu1*100:.2f}")
print(f"BLEU-2: {bleu2*100:.2f}")
print(f"BLEU-3: {bleu3*100:.2f}")
print(f"BLEU-4: {bleu4*100:.2f}")
print(f"METEOR: {avg_meteor*100:.2f}")
print("-"*40)

In [ ]:
# === MODIFIED: Save evaluation results to CSV (with emotion columns) ===

results_df = pd.DataFrame(evaluation_results)
results_df.to_csv("/kaggle/working/evaluation_results.csv", index=False)
print("\nEvaluation results saved to: /kaggle/working/evaluation_results.csv")
print("  Columns: image_path, ground_truth, prediction, meteor_score, detected_emotion, emotion_aware_caption")

# === UNCHANGED: Save summary scores (BASE captions only) ===
summary_df = pd.DataFrame({
    'Metric': ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'METEOR'],
    'Score': [bleu1*100, bleu2*100, bleu3*100, bleu4*100, avg_meteor*100]
})
summary_df.to_csv("/kaggle/working/evaluation_summary.csv", index=False)
print("Summary scores saved to: /kaggle/working/evaluation_summary.csv")

print("\nSummary (BASE captions only):")
print(summary_df.to_string(index=False))

In [ ]:
# === ADDED: Display Best 3 and Worst 3 METEOR scores with images ===

# Sort results by METEOR score
sorted_results = sorted(evaluation_results, key=lambda x: x['meteor_score'], reverse=True)

# Get best 3 and worst 3
best_3 = sorted_results[:3]
worst_3 = sorted_results[-3:]

print("\n" + "="*60)
print("TOP 3 BEST METEOR SCORES")
print("="*60)

for i, result in enumerate(best_3, 1):
    print(f"\n--- Best #{i} (METEOR: {result['meteor_score']*100:.2f}) ---")
    print(f"Image: {result['image_path']}")
    print(f"Ground Truth: {result['ground_truth']}")
    print(f"Base Caption: {result['prediction']}")
    print(f"Detected Emotion: {result['detected_emotion']}")
    print(f"Emotion-Aware Caption: {result['emotion_aware_caption']}")
    
    # Display image
    try:
        img = Image.open(result['image_path']).convert("RGB")
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Best #{i} | METEOR: {result['meteor_score']*100:.2f}\nEmotion: {result['detected_emotion']}\n{result['emotion_aware_caption'][:70]}...", fontsize=10)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not display image: {e}")

print("\n" + "="*60)
print("TOP 3 WORST METEOR SCORES")
print("="*60)

for i, result in enumerate(worst_3, 1):
    print(f"\n--- Worst #{i} (METEOR: {result['meteor_score']*100:.2f}) ---")
    print(f"Image: {result['image_path']}")
    print(f"Ground Truth: {result['ground_truth']}")
    print(f"Base Caption: {result['prediction']}")
    print(f"Detected Emotion: {result['detected_emotion']}")
    print(f"Emotion-Aware Caption: {result['emotion_aware_caption']}")
    
    # Display image
    try:
        img = Image.open(result['image_path']).convert("RGB")
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Worst #{i} | METEOR: {result['meteor_score']*100:.2f}\nEmotion: {result['detected_emotion']}\n{result['emotion_aware_caption'][:70]}...", fontsize=10)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not display image: {e}")

## === NEW: Test on Custom Images (testimages dataset) ===

In [ ]:
# === NEW: Test on custom images from /kaggle/input/testimages ===

print("\n" + "="*60)
print("TESTING ON CUSTOM IMAGES (testimages dataset)")
print("="*60)

# Path to custom images
custom_images_path = "/kaggle/input/testingimages"

# Check if custom images directory exists
if not os.path.exists(custom_images_path):
    print(f"\n⚠️ Custom images directory not found: {custom_images_path}")
    print("Please upload images to 'testimages' dataset in Kaggle.")
    custom_results = []
else:
    # Get all image files from the directory (any extension)
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.gif']
    custom_image_files = []
    
    for file in os.listdir(custom_images_path):
        if any(file.lower().endswith(ext) for ext in image_extensions):
            custom_image_files.append(file)
    
    print(f"\nFound {len(custom_image_files)} custom images")
    print(f"Image files: {custom_image_files}")
    
    custom_results = []
    
    for i, img_file in enumerate(custom_image_files, 1):
        img_path = os.path.join(custom_images_path, img_file)
        
        try:
            print(f"\n[{i}/{len(custom_image_files)}] Processing: {img_file}")
            
            # Generate BASE caption
            base_caption = caption_one(img_path, max_len=40)
            
            # Detect emotion and create emotion-aware caption
            emotion, emotion_aware_caption = generate_emotion_aware_caption(img_path, base_caption)
            
            print(f"  Base Caption: {base_caption}")
            print(f"  Detected Emotion: {emotion}")
            print(f"  Emotion-Aware Caption: {emotion_aware_caption}")
            print("-" * 70)
            
            # Display image
            img = Image.open(img_path).convert("RGB")
            plt.figure(figsize=(8, 8))
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"{img_file}\nEmotion: {emotion}\n{emotion_aware_caption}", fontsize=10)
            plt.tight_layout()
            plt.show()
            
            # Store result
            custom_results.append({
                'image': img_file,
                'base_caption': base_caption,
                'emotion': emotion,
                'aware_caption': emotion_aware_caption
            })
            
        except Exception as e:
            print(f"[ERROR] Error processing {img_file}: {e}")

# === Save results2.csv ===
if custom_results:
    custom_results_df = pd.DataFrame(custom_results)
    custom_results_df.to_csv("/kaggle/working/results2.csv", index=False)
    print(f"\n✓ Custom images results saved to /kaggle/working/results2.csv")
    print(f"  Columns: image, base_caption, emotion, aware_caption")
    print(f"  Total images processed: {len(custom_results)}")
else:
    print(f"\n⚠️ No custom images were processed.")
    print(f"  To test custom images, upload them to /kaggle/input/testingimages")

In [ ]:
# === FINAL SUMMARY ===

print("\n" + "="*60)
print("FINAL EVALUATION SUMMARY")
print("="*60)
print(f"\nModel: ViT (vit_base_patch16_224) + GPT-2 Decoder")
print(f"Dataset: Flickr30k")
print(f"Train/Test Split: 80%/20%")
print(f"Training Epochs: 6 (2 frozen + 4 fine-tuned)")
print(f"Test Set Size: {len(test_data_df)} images")
print(f"\n### PAPER-ALIGNED HYPERPARAMETERS ###")
print(f"Context Length: 50 (paper: 20-50)")
print(f"Dropout: 0.5 (paper: 0.1-0.5)")
print(f"\n--- Evaluation Metrics (BASE CAPTIONS ONLY) ---")
print(f"BLEU-1: {bleu1*100:.2f}")
print(f"BLEU-2: {bleu2*100:.2f}")
print(f"BLEU-3: {bleu3*100:.2f}")
print(f"BLEU-4: {bleu4*100:.2f}")
print(f"METEOR: {avg_meteor*100:.2f}")
print("="*60)
print("\n### FILES GENERATED ###")
print("1. image_caption_model.pt - Trained model")
print("2. evaluation_results.csv - Test set results with emotion (6 columns):")
print("     - image_path, ground_truth, prediction, meteor_score,")
print("       detected_emotion, emotion_aware_caption")
print("3. evaluation_summary.csv - BASE caption metrics only (unchanged)")
print("4. results2.csv - Custom images with emotion (4 columns):")
print("     - image, base_caption, emotion, aware_caption")
print("="*60)